# Real Out-of-Sample Strategy Validation, Short-Term (Weekly) Momentum Regime

Epic 16: extends Epic 15's real walk-forward + pre-registered-holdout validation
(`notebooks/research/out_of_sample_validation.ipynb`) to the **weekly** momentum regime.
Epic 15 only validated `portfolio1`'s monthly regime; `docs/STRATEGY_THEORY.md` still flagged
short-term/weekly momentum as "this project's own walk-forward tooling hasn't specifically
stress-tested it either." `portfolio2` (58 tickers) and `portfolio3` (11 tickers) both run this
exact weekly regime live today (`holding_period: 0.25`, `lookback_period: 1.0`), so this closes
a real, live gap, not a hypothetical one.

**Zero new code needed for the report/CI layer**: `backtest/momentum_backtest.py`'s
`_build_report()` always resamples to month-end (`daily.resample("ME").last()`) regardless of
`holding_period`, so Epic 15's `run_walk_forward_lookback_search()` and `bootstrap_sharpe_ci()`
work unchanged against a weekly-rebalanced equity curve, only the `lookback_candidates` grid and
the loaded portfolio config differ from Epic 15's notebook.

**Same scope boundary as Epic 13/14/15**: reuses Epic 13's already-cached long-history ETF
proxy universe (`crash_test_daily_prices.pkl`, 17 tickers, 2005-01-03 through today, no new
fetch needed) with `portfolio2`'s REAL weekly risk regime (`holding_period=0.25`,
`lookback_period=1.0` baseline, `use_correlation_penalty=True`, plus `default_risk`'s
`top_n=10`, `stop_loss_pct=0.12`, regime filter, vol targeting) applied to it. This validates
whether the WEEKLY STRATEGY MECHANICS generalize out of sample on a representative universe,
not a claim about `portfolio2`'s exact 58-ticker live universe.

Entirely backtest/research-only, no live paper account touched at all.

In [ ]:
# Package is pip-installed editable, no sys.path hacking needed
from dataclasses import replace

import pandas as pd

from momentum_trading.daily_runner import load_config
from momentum_trading.core import functions_quant_extensions as fnx
from momentum_trading.core.strategy_signals import generate_strategy_monthly_picks
from momentum_trading.backtest.momentum_backtest import run_custom_backtest

## 1. Load the real config and the cached proxy-universe price history

In [ ]:
config = load_config()
portfolio2_cfg = config["portfolios_resolved"]["portfolio2"]["cfg"]  # real weekly config

# Confirmed via a direct yfinance check (Epic 13) that all 17 have real data back to 2005.
PROXY_TICKERS = [
    "SPY", "QQQ", "DIA", "XLK", "XLF", "XLE", "XLI", "XLP", "XLU", "XLV", "XLY",
    "GLD", "TLT", "IEF", "SHY", "LQD", "IWM",
]

# Built by Epic 13's fetch step, relative to this notebook's own directory.
daily_prices = pd.read_pickle("crash_test_daily_prices.pkl")
print(daily_prices.shape, daily_prices.index.min(), "->", daily_prices.index.max())
print(f"strategy_type={portfolio2_cfg.strategy_type} top_n={portfolio2_cfg.top_n} "
      f"holding_period={portfolio2_cfg.holding_period} lookback_period={portfolio2_cfg.lookback_period} "
      f"stop_loss_pct={portfolio2_cfg.stop_loss_pct} use_regime_filter={portfolio2_cfg.use_regime_filter}")

## 2. Pre-registered train / holdout split

Same split date as Epic 15's monthly-regime notebook, for direct comparability. Commit to this
split BEFORE any tuning. `train` is used for walk-forward parameter search below; `holdout` is
not touched (not even glanced at) until the single final evaluation.

In [ ]:
train, holdout = fnx.pre_registered_split(daily_prices, split_date="2015-01-01")
print(f"Train:   {train.index.min().date()} to {train.index.max().date()} ({len(train)} rows)")
print(f"Holdout: {holdout.index.min().date()} to {holdout.index.max().date()} ({len(holdout)} rows)")

## 3. Walk-forward lookback search, TRAIN only

Same real-engine walk-forward search as Epic 15 (`run_walk_forward_lookback_search()`), but
`lookback_candidates` are in week-quarter units (the same `round(x * 4)` convention
`config.yaml`'s own `portfolio2.risk_overrides` documents): `0.5`/`0.75`/`1.0`/`1.5`/`2.0` ->
2/3/4/6/8 weeks. `portfolio2`'s real config is otherwise unchanged (weekly `holding_period`,
`use_correlation_penalty`, `top_n`, `stop_loss_pct`, regime filter, vol targeting).

In [ ]:
LOOKBACK_CANDIDATES = [0.5, 0.75, 1.0, 1.5, 2.0]  # week-quarters -> 2/3/4/6/8 weeks

wf_results = fnx.run_walk_forward_lookback_search(
    train, PROXY_TICKERS, portfolio2_cfg,
    lookback_candidates=LOOKBACK_CANDIDATES,
    train_years=4, test_years=1, step_years=1,
    metric="Sharpe",
)
pd.set_option("display.max_columns", None)
print(wf_results.to_string())

## 4. Choose the most robust lookback, evaluate on HOLDOUT exactly once

The most FREQUENTLY chosen lookback across folds (not just the single highest test_Sharpe
fold, which would just be cherry-picking one lucky fold), evaluated on `holdout` ONE time.
This is the project's first genuine out-of-sample number for the weekly regime, report it
as-is.

In [ ]:
chosen_lookback = float(wf_results["chosen_lookback"].mode().iloc[0])
print(f"Most frequently chosen lookback across {len(wf_results)} folds: {chosen_lookback} (week-quarters)")
print(f"Mean train_Sharpe across folds: {wf_results['train_Sharpe'].mean():.2f}")
print(f"Mean test_Sharpe across folds:  {wf_results['test_Sharpe'].mean():.2f}")

holdout_cfg = replace(portfolio2_cfg, lookback_period=chosen_lookback)
holdout_start = holdout.index.min()

# Bounded only at the end (Epic 14's own lesson): use the FULL daily_prices panel for real
# lookback history before holdout_start, filter the resulting PICKS down to the holdout window.
picks_full = generate_strategy_monthly_picks(
    daily_prices, PROXY_TICKERS, holdout_cfg, chosen_lookback, holdout_cfg.top_n,
)
picks_holdout = picks_full[picks_full.index >= holdout_start]

holdout_bt = run_custom_backtest(picks_holdout, daily_prices, **holdout_cfg.__dict__)
holdout_tearsheet = holdout_bt.attrs.get("tearsheet", {})

print(f"\nHOLDOUT ({holdout.index.min().date()} to {holdout.index.max().date()}), "
      f"chosen_lookback={chosen_lookback}, reported ONCE:")
for k, v in holdout_tearsheet.items():
    print(f"  {k}: {v}")

## 5. Block-bootstrap confidence interval on the holdout Sharpe

A confidence interval, not just a single point estimate from one historical path. Same
`block_size=6` (months, `_build_report()` still buckets to month-end regardless of the
underlying weekly rebalance cadence) as Epic 15's monthly-regime notebook.

In [ ]:
holdout_monthly_returns = holdout_bt["Portfolio Monthly Return"].dropna()

ci = fnx.bootstrap_sharpe_ci(holdout_monthly_returns, n_bootstrap=2000, block_size=6)
print(f"Holdout Sharpe point estimate: {ci['point_estimate']:.2f}")
print(f"{int(ci['confidence']*100)}% CI: [{ci['ci_low']:.2f}, {ci['ci_high']:.2f}]")
print(f"% of bootstrap samples with positive Sharpe: {ci['pct_bootstrap_samples_positive']*100:.1f}%")